# Partitioned LDSC

Goal: run partitioned LDSC in the refactored codebase by building query annotations, computing baseline-plus-query LD scores from an R2-table reference panel, and regressing one trait on that partitioned design.

Path-token rules used below:
- use `@` for chromosome suites such as `baseline.@`
- use globs when the filenames do not follow the simple chromosome-suffix convention
- scalar inputs still resolve to exactly one file
- output prefixes remain literal destinations


# global config

In [1]:
import os
print(f"Current working directory is {os.getcwd()}")

Current working directory is /Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry/.worktrees/restructure/refactor/tutorials


In [2]:
from pathlib import Path

import sys
sys.path.append("/Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry/.worktrees/restructure/refactor/src")

from ldsc import AnnotationBuildConfig, AnnotationBuilder, AnnotationSourceSpec, CommonConfig, LDScoreCalculator, LDScoreConfig, MungeConfig, RawSumstatsSpec, RefPanelLoader, RefPanelSpec, RegressionConfig, RegressionRunner, SumstatsMunger, run_bed_to_annot

# configure common settings
DATA_DIR = Path("../../result_consistency_checks")
common = CommonConfig(snp_identifier="chr_pos", genome_build="hg19")


# Processing annotation files

In [3]:
# convert BED files to annotation files
run_bed_to_annot(
    bed_files=str(DATA_DIR / "resources" / "raw_annot" / "*.bed"),
    baseline_annot_dir=DATA_DIR / "resources" / "baseline_v1.2",
    output_dir=DATA_DIR / "new_output" / "annot_processed",
)

INFO: Inferred canonical field 'POS' from input column 'BP' in ../../result_consistency_checks/resources/baseline_v1.2/baseline.1.annot.
/Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry/.worktrees/restructure/refactor/src/ldsc/_kernel/annotation.py:858: UserWarning: Inferred canonical field 'POS' from input column 'BP' in ../../result_consistency_checks/resources/baseline_v1.2/baseline.1.annot.
  column = resolve_required_column(header, spec, context=str(path))
INFO: Inferred canonical field 'POS' from input column 'BP' in ../../result_consistency_checks/resources/baseline_v1.2/baseline.1.annot.gz.
/Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry/.worktrees/restructure/refactor/src/ldsc/_kernel/annotation.py:858: UserWarning: Inferred canonical field 'POS' from input column 'BP' in ../../result_consistency_checks/resources/basel

PosixPath('../../result_consistency_checks/new_output/annot_processed')

# Calculate ld scores

In [4]:
# create annotation bundle
annotation_bundle = AnnotationBuilder(common, AnnotationBuildConfig()).run(
    AnnotationSourceSpec(
        baseline_annot_paths=str(DATA_DIR / "resources" / "baseline_v1.2" / "baseline.@"),
        query_annot_paths=str(DATA_DIR / "new_output" / "annot_processed" / "query.@"),
    )
)

In [5]:
# build ref panel from R2 table
ref_panel = RefPanelLoader(common).load(
    RefPanelSpec(
        backend="parquet_r2",
        r2_table_paths=str(DATA_DIR / "resources" / "ref_panel" / "R2_table" / "hm3_only" / "EUR_chr*_LD_MAF0.01_SNPonly_hm3only.parquet"),
        chromosomes=tuple(annotation_bundle.chromosomes),
    )
)

In [6]:
# calculate ld scores with the workflow-layer API
ldscore_result = LDScoreCalculator().run(
    annotation_bundle=annotation_bundle,
    ref_panel=ref_panel,
    ldscore_config=LDScoreConfig(ld_wind_cm=1.0),
    common_config=common,
)

runner = RegressionRunner(common, RegressionConfig())

INFO: Inferred canonical field 'hg38_pos_1' from input column 'hg38_bp1'.
/Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry/.worktrees/restructure/refactor/src/ldsc/column_inference.py:196: UserWarning: Inferred canonical field 'hg38_pos_1' from input column 'hg38_bp1'.
  return {spec.canonical: resolve_required_column(columns, spec, context=context) for spec in specs}
INFO: Inferred canonical field 'hg38_pos_2' from input column 'hg38_bp2'.
/Users/wenbinwu/Documents_local/Research/SullivanLab/LDSC/repos/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry_workspace/ldsc_py3_Jerry/.worktrees/restructure/refactor/src/ldsc/column_inference.py:196: UserWarning: Inferred canonical field 'hg38_pos_2' from input column 'hg38_bp2'.
  return {spec.canonical: resolve_required_column(columns, spec, context=context) for spec in specs}
INFO: Inferred canonical field 'hg19_pos_1' from input column 'hg19_bp_1'.
/Users/wenbinwu/Documen

ValueError: Could not infer a R2 rsID_1 column for canonical field 'rsID_1' from: chr, hg19_bp_1, hg19_bp_2

Since the annotation files and reference panel do not cover exactly the same SNP universe, the LD-score run can only use SNPs that exist in both:
- the annotation bundle
- the R2 table

In [ ]:
# compare the number of SNPs per chromosome before and after constructing ref_panel
before = annotation_bundle.metadata["CHR"].value_counts().sort_index()
after = ldscore_result.reference_metadata["CHR"].value_counts().sort_index()

comparison = (
    before.rename("annotation_bundle")
    .to_frame()
    .join(after.rename("retained_in_ldscore"), how="outer")
    .fillna(0)
    .astype(int)
)
comparison["dropped"] = comparison["annotation_bundle"] - comparison["retained_in_ldscore"]
comparison


# Munge sumstats

In [ ]:
# munge summary statistics
sumstats_path = DATA_DIR / "resources" / "sumstats" / "pgc-mdd2025_no23andMe-noUKBB_eur_v3-49-24-11_clean.tsv"
out_prefix = DATA_DIR / "new_output" / "sumstats" / "mdd2025"
sumstats = SumstatsMunger().run(
    RawSumstatsSpec(
        path=sumstats_path,
        trait_name="mdd2025",
    ),
    MungeConfig(
        out_prefix=out_prefix
    ),
    common,
)

# PLDSC

In [ ]:
# estimate partitioned heritability for each query
partitioned = runner.estimate_partitioned_h2_batch(
    sumstats,
    ldscore_result,
    annotation_bundle,
)

# save results
# partitioned.to_csv("tutorial_outputs/partitioned_h2.tsv", sep="\t", index=False)
print(partitioned)